# Orchestrator

Consolidates `penrs_arbiter.py` and `penrs_pipeline.py` into a single notebook. The **ArbiterAgent** aggregates and validates worker scores, detects mandatory contradictions, and produces a weighted composite signal. The **MasterAgent** and `run_penrs` entry-point drive the full end-to-end PENRS pipeline, persisting a JSON report for each run.

In [ ]:
from __future__ import annotations

import asyncio
import json
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Sequence

from pydantic import BaseModel, ConfigDict, Field

## Arbiter Helpers

Module-level constants (`ARBITER_SYSTEM_PROMPT`, `_MANDATORY_CONTRADICTIONS`) and pure helper functions used exclusively by `ArbiterAgent`: field validation, float coercion, score normalisation, star-rating derivation, narrative text collection, and contradiction detection.

In [ ]:
ARBITER_SYSTEM_PROMPT = (
    "You are the Lead Portfolio Manager reviewing all worker outputs for consistency.\n"
    "You must flag mandatory contradictions when present:\n"
    "- Lipstick on a Pig\n"
    "- Bailing Out\n"
    "- Dilute and Delay\n"
    "Return strictly valid JSON."
)

_MANDATORY_CONTRADICTIONS: tuple[dict[str, Any], ...] = (
    {
        "name": "Lipstick on a Pig",
        "severity": "High",
        "patterns": (r"\blipstick on a pig\b",),
    },
    {
        "name": "Bailing Out",
        "severity": "High",
        "patterns": (r"\bbailing out\b", r"\bbail[- ]?out\b"),
    },
    {
        "name": "Dilute and Delay",
        "severity": "Medium",
        "patterns": (r"\bdilute and delay\b",),
    },
)


def _require_field(payload: dict[str, Any], field_name: str, path: str) -> Any:
    if field_name not in payload:
        raise ValueError(f"Missing required field: {path}")
    return payload[field_name]


def _coerce_float(value: Any, field_path: str) -> float:
    try:
        return float(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"Field '{field_path}' must be numeric") from exc


def _normalize_score(score: float) -> float:
    return max(-1.0, min(1.0, score))


def _derive_star_rating(signal_density: float) -> int:
    if signal_density >= 0.85:
        return 5
    if signal_density >= 0.65:
        return 4
    if signal_density >= 0.45:
        return 3
    if signal_density >= 0.25:
        return 2
    return 1


def _extract_star_rating(result: dict[str, Any], signal_density: float) -> int:
    rating = result.get("star_rating")
    if rating is None:
        return _derive_star_rating(signal_density)
    rating_value = _coerce_float(rating, "result.star_rating")
    return int(max(1, min(5, round(rating_value))))


def _collect_narrative_text(result: dict[str, Any]) -> str:
    fragments: list[str] = []
    for key in ("summary", "thesis", "narrative", "analysis"):
        value = result.get(key)
        if isinstance(value, str) and value.strip():
            fragments.append(value.strip())
    return " ".join(fragments)


def _detect_mandatory_contradictions(narrative_text: str) -> list[dict[str, Any]]:
    normalized_text = narrative_text.lower()
    contradictions: list[dict[str, Any]] = []
    for rule in _MANDATORY_CONTRADICTIONS:
        matched_phrase = None
        for pattern in rule["patterns"]:
            match = re.search(pattern, normalized_text, flags=re.IGNORECASE)
            if match:
                matched_phrase = match.group(0)
                break

        contradictions.append(
            {
                "name": rule["name"],
                "severity": rule["severity"],
                "flagged": matched_phrase is not None,
                "evidence": matched_phrase,
            }
        )
    return contradictions

## ArbiterAgent

Aggregates worker results into a single weighted score. Skips non-`available` workers, applies star-rating-weighted scoring, and checks narrative text for mandatory contradiction patterns.

In [ ]:
class ArbiterAgent:
    def __init__(self, system_prompt: str = ARBITER_SYSTEM_PROMPT) -> None:
        self.system_prompt = system_prompt

    def _validate_worker_result(self, worker_result: dict[str, Any]) -> None:
        if not isinstance(worker_result, dict):
            raise ValueError("Each worker result must be a JSON object")

        _require_field(worker_result, "status", "status")
        worker = _require_field(worker_result, "worker", "worker")
        result = _require_field(worker_result, "result", "result")

        if not isinstance(worker, dict):
            raise ValueError("Field 'worker' must be an object")
        if not isinstance(result, dict):
            raise ValueError("Field 'result' must be an object")

        _require_field(worker, "name", "worker.name")
        _require_field(worker, "weight", "worker.weight")
        _require_field(worker, "signal_density", "worker.signal_density")
        _require_field(result, "score", "result.score")

    def evaluate(self, worker_results: list[dict[str, Any]]) -> dict[str, Any]:
        worker_scores: list[dict[str, Any]] = []
        narrative_fragments: list[str] = []
        total_effective_weight = 0.0
        weighted_score_sum = 0.0

        for worker_result in worker_results:
            self._validate_worker_result(worker_result)
            if worker_result.get("status") != "available":
                continue

            worker = worker_result["worker"]
            result = worker_result["result"]

            name = str(worker["name"])
            base_weight = _coerce_float(worker["weight"], "worker.weight")
            signal_density = _coerce_float(worker["signal_density"], "worker.signal_density")
            raw_score = _coerce_float(result["score"], "result.score")
            normalized_score = _normalize_score(raw_score)
            star_rating = _extract_star_rating(result, signal_density)
            effective_weight = round(base_weight * (star_rating / 5.0), 10)
            weighted_score = round(normalized_score * effective_weight, 10)

            total_effective_weight += effective_weight
            weighted_score_sum += weighted_score
            narrative_fragments.append(_collect_narrative_text(result))

            worker_scores.append(
                {
                    "name": name,
                    "raw_score": raw_score,
                    "normalized_score": normalized_score,
                    "weight": base_weight,
                    "star_rating": star_rating,
                    "effective_weight": effective_weight,
                    "weighted_score": weighted_score,
                }
            )

        weighted_score = 0.0
        if total_effective_weight > 0:
            weighted_score = weighted_score_sum / total_effective_weight
            weighted_score = _normalize_score(weighted_score)
            weighted_score = round(weighted_score, 10)

        contradictions = _detect_mandatory_contradictions(" ".join(narrative_fragments))

        return {
            "status": "available",
            "arbiter_role": "Lead Portfolio Manager",
            "system_prompt": self.system_prompt,
            "worker_scores": worker_scores,
            "weighted_score": weighted_score,
            "contradictions": contradictions,
        }

## Pipeline Helpers

Async-compatibility shim (`_maybe_await`), worker introspection and invocation helpers, a concurrent fan-out runner (`run_all_workers`), arbiter input filtering (`_build_arbiter_input`), safe error-wrapping arbiter call (`_evaluate_with_arbiter`), and a filename sanitiser (`_safe_filename_component`).

In [ ]:
async def _maybe_await(value: Any) -> Any:
    if hasattr(value, "__await__"):
        return await value
    return value


def _coerce_float_or_zero(value: Any) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return 0.0


def _worker_identity(worker: Any) -> dict[str, Any]:
    return {
        "name": str(getattr(worker, "name", worker.__class__.__name__)),
        "weight": _coerce_float_or_zero(getattr(worker, "weight", 0.0)),
        "signal_density": _coerce_float_or_zero(getattr(worker, "signal_density", 0.0)),
    }


async def _invoke_worker(worker: Any, ticker: str, date_from: str, date_to: str) -> Any:
    run_callable = getattr(worker, "run", None)
    if run_callable is None or not callable(run_callable):
        raise TypeError("Worker must expose a callable run(ticker, date_from, date_to)")
    return await _maybe_await(run_callable(ticker, date_from, date_to))


async def run_all_workers(
    workers: Sequence[Any],
    ticker: str,
    date_from: str,
    date_to: str,
) -> list[dict[str, Any]]:
    worker_list = list(workers)
    if not worker_list:
        return []

    coroutines = [_invoke_worker(worker, ticker, date_from, date_to) for worker in worker_list]
    raw_results = await asyncio.gather(*coroutines, return_exceptions=True)

    worker_results: list[dict[str, Any]] = []
    for worker, raw_result in zip(worker_list, raw_results):
        if isinstance(raw_result, Exception):
            worker_results.append(
                {
                    "status": "error",
                    "worker": _worker_identity(worker),
                    "ticker": ticker,
                    "date_from": date_from,
                    "date_to": date_to,
                    "error": str(raw_result),
                }
            )
            continue

        if not isinstance(raw_result, dict):
            worker_results.append(
                {
                    "status": "error",
                    "worker": _worker_identity(worker),
                    "ticker": ticker,
                    "date_from": date_from,
                    "date_to": date_to,
                    "error": "Worker returned non-dict result",
                }
            )
            continue

        worker_results.append(raw_result)

    return worker_results


def _build_arbiter_input(worker_results: list[dict[str, Any]]) -> list[dict[str, Any]]:
    arbiter_input: list[dict[str, Any]] = []
    for worker_result in worker_results:
        if worker_result.get("status") != "available":
            continue
        if not isinstance(worker_result.get("worker"), dict):
            continue
        if not isinstance(worker_result.get("result"), dict):
            continue
        arbiter_input.append(worker_result)
    return arbiter_input


def _evaluate_with_arbiter(arbiter: ArbiterAgent, worker_results: list[dict[str, Any]]) -> dict[str, Any]:
    arbiter_input = _build_arbiter_input(worker_results)
    if not arbiter_input:
        return {
            "status": "not_available",
            "worker_scores": [],
            "weighted_score": 0.0,
            "contradictions": [],
        }

    try:
        return arbiter.evaluate(arbiter_input)
    except Exception as exc:
        return {
            "status": "error",
            "error": str(exc),
            "worker_scores": [],
            "weighted_score": 0.0,
            "contradictions": [],
        }


def _safe_filename_component(value: str) -> str:
    sanitized = "".join(char if (char.isalnum() or char in {"-", "_"}) else "_" for char in value)
    sanitized = sanitized.strip("_")
    return sanitized or "unknown"

## MasterAgent

Programmatic synthesis layer that wraps the arbiter's weighted score into the top-level `master` result dict, reporting available-worker counts and date range alongside the `final_score`.

In [ ]:
class MasterAgent:
    def synthesize(
        self,
        *,
        ticker: str,
        date_from: str,
        date_to: str,
        worker_results: list[dict[str, Any]],
        arbiter_result: dict[str, Any],
    ) -> dict[str, Any]:
        available_workers = sum(1 for result in worker_results if result.get("status") == "available")
        weighted_score = arbiter_result.get("weighted_score", 0.0)
        if not isinstance(weighted_score, (int, float)):
            weighted_score = 0.0

        return {
            "status": "available",
            "model": "programmatic_master_v1",
            "ticker": ticker,
            "date_range": {"from": date_from, "to": date_to},
            "final_score": float(weighted_score),
            "available_worker_count": available_workers,
            "total_worker_count": len(worker_results),
        }

## PENRSReport Schema

Pydantic model for the persisted JSON report. Uses `extra="allow"` so future fields can be added without breaking validation.

In [ ]:
class PENRSReport(BaseModel):
    model_config = ConfigDict(extra="allow")

    ticker: str
    date_from: str
    date_to: str
    generated_at: str
    worker_results: list[dict[str, Any]] = Field(default_factory=list)
    arbiter: dict[str, Any]
    master: dict[str, Any]
    report_path: str

## run_penrs

The main entry-point coroutine. Fans out workers, runs the arbiter, synthesises the master result, and writes a timestamped JSON report to `report_dir`.

In [ ]:
async def run_penrs(
    ticker: str,
    date_from: str,
    date_to: str,
    *,
    workers: Sequence[Any] | None = None,
    arbiter: ArbiterAgent | None = None,
    master: MasterAgent | None = None,
    report_dir: str | Path = "penrs_reports",
    now: datetime | None = None,
) -> dict[str, Any]:
    worker_results = await run_all_workers(list(workers or []), ticker, date_from, date_to)

    arbiter_agent = arbiter or ArbiterAgent()
    arbiter_result = _evaluate_with_arbiter(arbiter_agent, worker_results)

    master_agent = master or MasterAgent()
    master_result_raw = master_agent.synthesize(
        ticker=ticker,
        date_from=date_from,
        date_to=date_to,
        worker_results=worker_results,
        arbiter_result=arbiter_result,
    )
    master_result = await _maybe_await(master_result_raw)
    if not isinstance(master_result, dict):
        master_result = {"status": "error", "error": "Master returned non-dict result"}

    generated_at = now or datetime.now(timezone.utc)
    generated_at_utc = generated_at.astimezone(timezone.utc)
    report_dir_path = Path(report_dir)
    report_dir_path.mkdir(parents=True, exist_ok=True)
    filename = (
        f"{_safe_filename_component(ticker)}_"
        f"{_safe_filename_component(date_from)}_"
        f"{_safe_filename_component(date_to)}_"
        f"{generated_at_utc.strftime('%Y%m%dT%H%M%SZ')}.json"
    )
    report_path = (report_dir_path / filename).resolve()

    report_payload = {
        "ticker": ticker,
        "date_from": date_from,
        "date_to": date_to,
        "generated_at": generated_at_utc.isoformat(),
        "worker_results": worker_results,
        "arbiter": arbiter_result,
        "master": master_result,
        "report_path": str(report_path),
    }
    validated_report = PENRSReport.model_validate(report_payload)
    serialized_report = validated_report.model_dump(mode="json")
    report_path.write_text(json.dumps(serialized_report, indent=2, ensure_ascii=True), encoding="utf-8")
    return serialized_report